In [1]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

file_path = project_root / "data" / "raw" / "Dataset Capstone.xlsx"

df_survey = pd.read_excel(
    file_path,
    sheet_name="customer_survey"
)

print("=== INFORMASI DATASET SURVEI ===")
print("Jumlah respons:", len(df_survey))
print("Jumlah kolom:", len(df_survey.columns))

display(df_survey.head())

=== INFORMASI DATASET SURVEI ===
Jumlah respons: 100
Jumlah kolom: 11


,response_id,submitted_at,respondent_type,information_needed,main_information,preferred_platform,information_difficulty,website_expectation,additional_feedback,consent,data_type
0,R001,2026-11-01,Pernah,"Jam Operasional, Lokasi, Fasilitas, Harga, Menu",Harga,Google Maps,2,Kontak WhatsApp atau media sosial agar pelangg...,Menu favorit atau best seller bisa diberi pena...,Setuju,actual
1,R002,2026-11-01,Pernah,"Promo/Event, Jam Operasional",Harga,Instagram,3,Fitur Ask AI untuk menjawab pertanyaan pelangg...,Tampilan website dibuat sederhana dan tidak me...,Setuju,actual
2,R003,2026-11-01,Pernah,"Promo/Event, Lokasi, Jam Operasional",Harga,Google Maps,5,Foto dan video suasana tempat serta fasilitas ...,Tidak ada saran tambahan.,Setuju,dummy
3,R004,2026-11-01,Pernah,"Harga, Fasilitas, Lokasi",Fasilitas,Instagram,3,"Informasi cabang, jam buka, menu, dan harga da...",Tidak ada saran tambahan.,Setuju,dummy
4,R005,2026-11-01,Belum Pernah,"Promo/Event, Menu, Lokasi, Jam Operasional, La...",Harga,Instagram,4,"Tampilan website yang sederhana, menarik, dan ...",Informasi menu dan harga sebaiknya ditampilkan...,Setuju,dummy


In [2]:
print("=== DISTRIBUSI JENIS DATA ===")

data_type_counts = (
    df_survey["data_type"]
    .value_counts(dropna=False)
)

display(data_type_counts)

print("\nPersentase jenis data:")

data_type_percentage = (
    df_survey["data_type"]
    .value_counts(normalize=True, dropna=False) * 100
).round(2)

display(data_type_percentage)

=== DISTRIBUSI JENIS DATA ===


data_type
dummy     98
actual     2
Name: count, dtype: int64


Persentase jenis data:


data_type
dummy     98.0
actual     2.0
Name: proportion, dtype: float64

In [3]:
print("=== PEMERIKSAAN MISSING VALUES ===")

missing_report = pd.DataFrame({
    "jumlah_kosong": df_survey.isna().sum(),
    "persentase_kosong": (
        df_survey.isna().mean() * 100
    ).round(2)
})

display(missing_report)

=== PEMERIKSAAN MISSING VALUES ===


,jumlah_kosong,persentase_kosong
response_id,0,0.0
submitted_at,0,0.0
respondent_type,0,0.0
information_needed,0,0.0
main_information,0,0.0
preferred_platform,0,0.0
information_difficulty,0,0.0
website_expectation,0,0.0
additional_feedback,1,1.0
consent,0,0.0


In [4]:
print("=== PEMERIKSAAN DUPLIKASI ===")

print(
    "Duplikasi seluruh baris:",
    df_survey.duplicated().sum()
)

print(
    "Duplikasi response_id:",
    df_survey["response_id"].duplicated().sum()
)

print(
    "Response ID kosong:",
    df_survey["response_id"].isna().sum()
)

=== PEMERIKSAAN DUPLIKASI ===
Duplikasi seluruh baris: 0
Duplikasi response_id: 0
Response ID kosong: 0


In [5]:
print("=== PEMERIKSAAN KONSISTENSI JAWABAN ===")

columns = [
    "respondent_type",
    "main_information",
    "preferred_platform",
    "information_difficulty",
    "consent"
]

for col in columns:
    print(f"\n=== {col} ===")
    display(
        df_survey[col].value_counts(dropna=False)
    )

=== PEMERIKSAAN KONSISTENSI JAWABAN ===

=== respondent_type ===


respondent_type
Pernah          71
Belum Pernah    29
Name: count, dtype: int64


=== main_information ===


main_information
Menu               30
Harga              23
Promo/Event        19
Lokasi             11
Jam Operasional    10
Fasilitas           7
Name: count, dtype: int64


=== preferred_platform ===


preferred_platform
Instagram        32
Website Resmi    22
TikTok           20
Google Maps      17
Lainnya           9
Name: count, dtype: int64


=== information_difficulty ===


information_difficulty
4    28
3    24
2    20
5    16
1    12
Name: count, dtype: int64


=== consent ===


consent
Setuju    100
Name: count, dtype: int64

In [6]:
print("=== CONSENT BERDASARKAN JENIS DATA ===")

consent_report = pd.crosstab(
    df_survey["data_type"].fillna("unknown"),
    df_survey["consent"].fillna("missing"),
    dropna=False
)

display(consent_report)

=== CONSENT BERDASARKAN JENIS DATA ===


consent,Setuju
data_type,
actual,2
dummy,98


In [7]:
print("=== VALIDASI LABEL DATA ===")

allowed_types = ["actual", "dummy"]

invalid_types = df_survey[
    ~df_survey["data_type"].isin(allowed_types)
]

print("Jumlah label tidak valid:", len(invalid_types))

print("\nJumlah berdasarkan jenis data:")
print(df_survey["data_type"].value_counts())

assert len(invalid_types) == 0, \
    "Terdapat label data_type tidak valid!"

assert df_survey["response_id"].is_unique, \
    "Terdapat response_id duplikat!"

print("\nValidasi label dan ID berhasil.")

=== VALIDASI LABEL DATA ===
Jumlah label tidak valid: 0

Jumlah berdasarkan jenis data:
data_type
dummy     98
actual     2
Name: count, dtype: int64

Validasi label dan ID berhasil.


In [8]:
reports_dir = project_root / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

assessment_summary = pd.DataFrame([
    {
        "metric": "total_responses",
        "value": len(df_survey)
    },
    {
        "metric": "actual_responses",
        "value": (df_survey["data_type"] == "actual").sum()
    },
    {
        "metric": "dummy_responses",
        "value": (df_survey["data_type"] == "dummy").sum()
    },
    {
        "metric": "duplicate_response_id",
        "value": df_survey["response_id"].duplicated().sum()
    },
    {
        "metric": "duplicate_rows",
        "value": df_survey.duplicated().sum()
    },
    {
        "metric": "missing_cells",
        "value": df_survey.isna().sum().sum()
    }
])

assessment_summary.to_csv(
    reports_dir / "survey_assessment_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

display(assessment_summary)

print("Laporan Data Assessing berhasil disimpan.")

,metric,value
0,total_responses,100
1,actual_responses,2
2,dummy_responses,98
3,duplicate_response_id,0
4,duplicate_rows,0
5,missing_cells,1


Laporan Data Assessing berhasil disimpan.
